In [ ]:
# Import packages, load the study configuration and the CHIP tables written by
# 11_chip_tables
from pathlib import Path

from SSc_Somatic import configure, load_config, load_study
from SSc_Somatic.chip import analysis as chip_funcs
from SSc_Somatic.chip import plots as chip_plots_funcs
from SSc_Somatic.loading import load_blood_sample_ids

# Study configuration. Point this at another configuration file to analyze a different cohort.
config = configure(load_config(Path('..') / 'config' / 'example_SSc.yaml'))
data = load_study(config, 'chip')

# Cohort labels and column names used below
status_col = config.require('cohort', 'status_column')
meta_id_col = config.require('cohort', 'metadata_sample_column')
case_label = config.require('cohort', 'case_label')
control_label = config.require('cohort', 'control_label')
subtype_a = config.require('cohort', 'subtypes')[0]
clinical = config.require('cohort', 'clinical_columns')
duration_col = clinical['disease_duration']
dlco_col = clinical['dlco_percent_predicted']
fvc_col = clinical['fvc_percent_predicted']
mrss_col = clinical['skin_score']

# Tables written by 11_chip_tables
wl_final_anno_df = data.whitelist_calls
wl_final_merged_df = data.whitelist_merged

In [ ]:
# Variables compared in the pairwise analyses below. Then keep cases and controls
# sequenced from blood, remove duplicate calls and report the table sizes.
variables_of_interest = [
    'gender', 'race', 'Ethnicity ', 'Age', 'Disease Duration ',
    'ana', 'anti-centromere', 'anti-topo', 'anti-RNA polymerase III',
    'MRSS', 'FVC %predicted', 'DLCO %Predicted', 'age_at_diagnosis'
]

wl_final_anno_df = wl_final_anno_df.loc[wl_final_anno_df[status_col].isin([case_label, control_label])]
wl_final_merged_df = wl_final_merged_df.loc[wl_final_merged_df[status_col].isin([case_label, control_label])]
blood_ids = load_blood_sample_ids(config)
print("Participants in the merged table after keeping cases and controls:",
      len(wl_final_merged_df[meta_id_col].unique()))
wl_final_anno_df = wl_final_anno_df.loc[wl_final_anno_df[meta_id_col].astype(str).isin(blood_ids)]
wl_final_merged_df = wl_final_merged_df.loc[wl_final_merged_df['sample_id'].astype(str).isin(blood_ids)]
wl_final_anno_df = wl_final_anno_df.drop_duplicates(subset=[meta_id_col, 'Start','Gene.refGene', 'VAF'], keep='first')
wl_final_merged_df = wl_final_merged_df.drop_duplicates(subset=['sample_id'], keep='first')
print("Participants with CHIP calls after keeping blood samples and removing duplicates:",
      len(wl_final_anno_df[meta_id_col].unique()))
print("Rows in the merged table after keeping blood samples and removing duplicates:",
      len(wl_final_merged_df))
print("Participants in the merged table:", len(wl_final_merged_df['sample_id'].unique()))

print("Annotated CHIP whitelist table (rows, columns):", wl_final_anno_df.shape)
print("Merged table (rows, columns):", wl_final_merged_df.shape)

In [ ]:
# Prepare the clinical variables: clean column names, add each participant's CHIP gene,
# and create tertile and binary groups for DLCO, FVC and MRSS
# Standardize column names first
print("Removing trailing spaces from column names:")
wl_final_anno_df = chip_funcs.standardize_column_names(wl_final_anno_df)
wl_final_merged_df = chip_funcs.standardize_column_names(wl_final_merged_df)
wl_final_merged_df['CHIP_gene'] = wl_final_merged_df['sample_id'].map(wl_final_anno_df.groupby('sample_id')['Gene.refGene'].first().to_dict())

# Create tertiles for each clinical measure, in both tables
tertile_metrics = {'DLCO': dlco_col, 'FVC': fvc_col, 'MRSS': mrss_col}
print("Splitting DLCO, FVC and MRSS into tertiles:")
tertile_info = {}
for prefix, column in tertile_metrics.items():
    wl_final_merged_df, tertile_info['merged', prefix] = chip_funcs.preprocess_clinical_tertiles(
        wl_final_merged_df, column, prefix=prefix)
for prefix, column in tertile_metrics.items():
    wl_final_anno_df, tertile_info['anno', prefix] = chip_funcs.preprocess_clinical_tertiles(
        wl_final_anno_df, column, prefix=prefix)

# Create binary DLCO classifications
print("Classifying DLCO into binary groups:")
wl_final_merged_df, dlco_binary_info = chip_funcs.preprocess_dlco_binary(wl_final_merged_df)
wl_final_anno_df, dlco_binary_info = chip_funcs.preprocess_dlco_binary(wl_final_anno_df)

In [ ]:
# Summarize tertile distributions, then analyze tertiles by CHIP status
print("Distribution of DLCO, FVC and MRSS tertiles:")
tertile_summaries = {
    prefix: chip_funcs.summarize_clinical_tertiles(wl_final_merged_df, column, prefix=prefix)
    for prefix, column in tertile_metrics.items()
}
print("DLCO, FVC and MRSS tertiles by CHIP status (chi-square tests):")
tertile_chip_analyses = {
    prefix: chip_funcs.analyze_tertiles_by_chip_status(wl_final_merged_df, column, prefix=prefix)
    for prefix, column in tertile_metrics.items()
}

In [ ]:
# Test whether autoantibody status is associated with CHIP (logistic regression)
print("Association of autoantibody status with CHIP:")
ana_results = chip_funcs.run_ana_analysis(wl_final_merged_df)

# Create forest plot
print("Figure: odds ratios with 95% confidence intervals for CHIP by autoantibody status")
fig_ana = chip_plots_funcs.plot_ana_forest(
    ana_results,
    output_filename=None
)

In [ ]:
# Compare DLCO, FVC and MRSS between CHIP status, case subtype and TET2 groups, and plot them
# Compute statistics
print("Comparing DLCO, FVC and MRSS between CHIP and TET2 groups (Kruskal-Wallis and "
      "Mann-Whitney U tests):")
metric_stats_results, plot_df, group_order = chip_funcs.compute_chip_tet2_group_statistics(
    wl_final_merged_df,
    metrics=[dlco_col, fvc_col, mrss_col],
    alpha=0.05
)

# Create plot
print("Figure: DLCO, FVC and MRSS by CHIP status, case subtype and TET2 group")
fig, axes = chip_plots_funcs.plot_chip_tet2_triple_metrics(
    plot_df,
    group_order,
    stats_results=metric_stats_results,
    output_filename=None
)

In [ ]:
# Count participants by CHIP gene
print("Number of participants by CHIP gene:")
wl_final_merged_df['CHIP_gene'].value_counts()

In [ ]:
# Pairwise comparisons within CHIP samples (uses wl_final_anno_df)
regression_genes = ['TET2', 'DNMT3A']
pairwise_results, pairwise_tables = {}, {}
for gene in regression_genes:
    print(f"Pairwise comparisons of clinical variables within {gene} CHIP carriers, with FDR correction:")
    pairwise_results[gene], pairwise_tables[gene] = chip_funcs.run_gene_pairwise_analysis_with_fdr(
        wl_final_anno_df,
        gene=gene,
        variables_to_test=variables_of_interest,
        percentile_thresholds=[50],
        gene_col='Gene.refGene',
        include_controls=False
    )

In [ ]:
# Regression analysis with clinical outcomes (uses wl_final_merged_df)
print(f"Regression of clinical outcomes on CHIP status for {', '.join(regression_genes)}, "
      "with FDR correction within each outcome:")
regression_results = {
    gene: chip_funcs.run_gene_within_metric_regression_analysis(wl_final_merged_df, gene=gene, gene_col='CHIP_gene')
    for gene in regression_genes
}

# Model performance summaries
for gene in regression_genes:
    print(f"Model performance for {gene}:")
    chip_funcs.create_model_performance_summary(regression_results[gene], gene=gene)

# Summarize pairwise results with direction
for gene in regression_genes:
    print(f"Summary of the {gene} pairwise comparisons, with direction of effect:")
    chip_funcs.summarize_results_with_direction(pairwise_results[gene])

In [ ]:
# Coefficient plots (individual genes)
print(f"Figures: regression coefficients with 95% confidence intervals across clinical "
      "outcomes and case subtypes, one figure for each of {', '.join(regression_genes)}")
coefficient_figures = {
    gene: chip_plots_funcs.plot_gene_within_metric_coefficients(regression_results[gene], gene=gene)
    for gene in regression_genes
}

In [ ]:
# Gene comparison panel (TET2 vs DNMT3A)
print(f"Figure: TET2 and DNMT3A carriers compared with participants without CHIP across "
      "clinical outcomes, {subtype_a} subtype (unadjusted p-values)")
fig_comparison = chip_plots_funcs.plot_gene_comparison_panel(
    regression_results['TET2'],
    regression_results['DNMT3A'],
    gene1_name='TET2',
    gene2_name='DNMT3A',
    comparison='vs_nonCHIP',
    subtype=subtype_a,
    use_fdr=False
)

In [ ]:
# Mean disease duration in TET2 CHIP carriers and all other participants
tet2_positive = wl_final_merged_df[wl_final_merged_df['CHIP_gene'] == 'TET2'][duration_col].dropna()
tet2_negative = wl_final_merged_df[wl_final_merged_df['CHIP_gene'] != 'TET2'][duration_col].dropna()

print("Mean disease duration in TET2 CHIP carriers and all other participants:")
print(f"TET2+ patients: mean duration = {tet2_positive.mean():.1f} years")
print(f"TET2- patients: mean duration = {tet2_negative.mean():.1f} years")

In [ ]:
# Check for disease duration confounding in TET2-FVC relationship
print("Checking whether disease duration confounds the association of TET2 CHIP with FVC:")
confounding_check = chip_funcs.check_chip_gene_confounding_quick(
    wl_final_merged_df,
    gene='TET2',
    outcome=fvc_col,
    confounder=duration_col,
    gene_col_name='CHIP_gene'
)